## Setup and MLflow Initialization
In this section I initialize the environment and connect the project to DagsHub and MLflow. This allows us to track every experiment remotely, ensuring that our model parameters and metrics are saved in a professional cloud registry.

In [76]:
import dagshub
import mlflow
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_squared_log_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor

# Set Up Tracking Environment and Remote Server
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML.mlflow")
mlflow.set_experiment("House_Price_Regression")


Initialized MLflow to track repo "GigiSichinava/ML"

Repository GigiSichinava/ML initialized!

<Experiment: artifact_location='mlflow-artifacts:/96e2751e965f44f1831b528fcd97e517', creation_time=1775950867574, experiment_id='1', last_update_time=1775950867574, lifecycle_stage='active', name='House_Price_Regression', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## Cleaning and Feature Engineering
Here I load the Kaggle dataset and focus on handling missing values (NaNs) and identifying numeric columns. By filling missing values with zeros I create a stable baseline for our regression models.

In [77]:
# Data Loading and Preparation
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Select only numeric columns and drop 'ID'
numeric_cols = train_df.select_dtypes(include=[np.number]).columns


## Feature Selection
For this experiment I filtered for numeric features and removed non-essential columns like 'Id'. I then split the data into 80% training and 20% validation sets to ensure we can properly test the model's performance on unseen data.

In [78]:
# Filtering for numeric columns only and removing IDs
X = train_df[numeric_cols].drop(['SalePrice', 'Id'], axis=1, errors='ignore').fillna(0)
y = train_df['SalePrice']

# Split into 80/20 portion
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Log Data Setup to DagsHub
with mlflow.start_run(run_name="Final_Data_Prep"):
    mlflow.log_param("train_shape", str(X_train.shape))
    mlflow.log_param("val_shape", str(X_val.shape))
    print(f"Setup Complete: Training on {X_train.shape[0]} houses.")


Setup Complete: Training on 1168 houses.
🏃 View run Final_Data_Prep at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/ef423868a4ff49c9976ed49688a9d142
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


## Model Training and MLflow Tracking
I created a automated function that trains different models (Linear Regression, XGBoost and Others), logs their performance metrics (RMSLE, RMSE, R2) to MLflow and pushes each model to the DagsHub Model Registry. This allows for seamless version control and ensures the best performing model can be easily retrieved for inference.

In [79]:
# The Metric Machine Function
def train_and_log(model, model_name):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        mae = mean_absolute_error(y_val, preds)
        r2 = r2_score(y_val, preds)
        rmsle = np.sqrt(mean_squared_log_error(y_val, np.clip(preds, 0, None)))

        # Log Metrics
        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2_score", r2)
        mlflow.log_metric("rmsle", rmsle)

        # Save to Model Registry
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name
        )

        # Printout
        print(f"Model logged: {model_name}")
        print(f"   RMSLE:          {rmsle:.4f}")
        print(f"   RMSE:           ${rmse:,.0f}")
        print(f"   MAE:            ${mae:,.0f}")
        print(f"   R2 Score:       {r2:.4f}")
        print("-" * 30)

        return model

##### Linear Regression

In [80]:
lr_model = train_and_log(LinearRegression(), "Linear_Regression")

2026/04/13 01:24:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:24:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Linear_Regression' already exists. Creating a new version of this model...
2026/04/13 01:24:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Linear_Regression, version 4
Created version '4' of model 'Linear_Regression'.


Model logged: Linear_Regression
   RMSLE:          0.1754
   RMSE:           $36,061
   MAE:            $22,187
   R2 Score:       0.8305
------------------------------
🏃 View run Linear_Regression at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/a777448723ad4d2a938570526e9912d4
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


##### Decision Tree

In [81]:
dt_model = train_and_log(DecisionTreeRegressor(random_state=42), "Decision_Tree")

2026/04/13 01:24:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:24:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Decision_Tree'.
2026/04/13 01:24:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Decision_Tree, version 1
Created version '1' of model 'Decision_Tree'.


Model logged: Decision_Tree
   RMSLE:          0.2321
   RMSE:           $43,128
   MAE:            $27,472
   R2 Score:       0.7575
------------------------------
🏃 View run Decision_Tree at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/12e646bdea5b4354bb3da6241d9f4408
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


##### Random Forest

In [82]:
rf_model = train_and_log(RandomForestRegressor(n_estimators=100, random_state=42), "Random_Forest")

2026/04/13 01:24:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:25:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Random_Forest'.
2026/04/13 01:25:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Random_Forest, version 1
Created version '1' of model 'Random_Forest'.


Model logged: Random_Forest
   RMSLE:          0.1541
   RMSE:           $29,194
   MAE:            $18,034
   R2 Score:       0.8889
------------------------------
🏃 View run Random_Forest at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/9f2860147a5642d18cc2bb2f4218d4cb
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


##### Tuned Random Forest

In [83]:
rf_tuned = train_and_log(RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42), "Tuned_Random_Forest")

2026/04/13 01:25:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:25:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Tuned_Random_Forest'.
2026/04/13 01:26:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Tuned_Random_Forest, version 1
Created version '1' of model 'Tuned_Random_Forest'.


Model logged: Tuned_Random_Forest
   RMSLE:          0.1546
   RMSE:           $28,708
   MAE:            $17,962
   R2 Score:       0.8926
------------------------------
🏃 View run Tuned_Random_Forest at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/5586d0d3004c42569a4907309181da5a
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


#### XGBoost
After comparing the results, XGBoost emerged as the "Champion" model with the lowest RMSLE (0.1434). This model was pushed to the production registry for the final inference task.

In [84]:
xgb_model = train_and_log(xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=5), "XGBoost")

2026/04/13 01:26:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:26:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGBoost' already exists. Creating a new version of this model...
2026/04/13 01:27:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost, version 3
Created version '3' of model 'XGBoost'.


Model logged: XGBoost
   RMSLE:          0.1434
   RMSE:           $26,946
   MAE:            $16,562
   R2 Score:       0.9053
------------------------------
🏃 View run XGBoost at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/ad760fc947c2457093411efeb23d831c
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


##### Ridge Regression

In [85]:
ridge_model = train_and_log(Ridge(alpha=1.0), "Ridge_Regression")

2026/04/13 01:27:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:27:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Ridge_Regression'.
2026/04/13 01:27:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Ridge_Regression, version 1
Created version '1' of model 'Ridge_Regression'.


Model logged: Ridge_Regression
   RMSLE:          0.1754
   RMSE:           $36,061
   MAE:            $22,177
   R2 Score:       0.8305
------------------------------
🏃 View run Ridge_Regression at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/a2e524c1796a4d71b6091caeb8e4e6d9
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


##### Gradient Boosting

In [86]:
gbr_model = train_and_log(GradientBoostingRegressor(n_estimators=100, random_state=42), "Gradient_Boosting")

2026/04/13 01:27:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:27:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Gradient_Boosting' already exists. Creating a new version of this model...
2026/04/13 01:28:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Gradient_Boosting, version 2
Created version '2' of model 'Gradient_Boosting'.


Model logged: Gradient_Boosting
   RMSLE:          0.1433
   RMSE:           $28,710
   MAE:            $17,552
   R2 Score:       0.8925
------------------------------
🏃 View run Gradient_Boosting at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/df36868bfb624abd986ad0e797e0d54a
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1


##### AdaBoost

In [87]:
ada_model = train_and_log(AdaBoostRegressor(n_estimators=100, random_state=42), "AdaBoost")

2026/04/13 01:28:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:28:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'AdaBoost' already exists. Creating a new version of this model...
2026/04/13 01:28:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AdaBoost, version 2
Created version '2' of model 'AdaBoost'.


Model logged: AdaBoost
   RMSLE:          0.2290
   RMSE:           $37,028
   MAE:            $26,136
   R2 Score:       0.8212
------------------------------
🏃 View run AdaBoost at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1/runs/62523702bef34a63b6b917450995b1e4
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML.mlflow/#/experiments/1
